# YOLOv5 vs YOLOv8 — Entraînement sur Pascal VOC
> **Projet final Computer Vision — Master IA & Big Data, GE-IT**
>

Ce notebook réalise les 10 runs d'entraînement du projet :
- 2 modèles (YOLOv5s, YOLOv8s)
- 5 stratégies d'augmentation (baseline, flip, rotation, mosaic, mixup)

**Prérequis :** Activer le GPU T4 dans *Exécution → Modifier le type d'exécution → GPU*

## Cellule 1 — Vérification du GPU

In [1]:
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA disponible : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU : {gpu.name}')
    print(f'VRAM : {gpu.total_memory / 1e9:.1f} Go')
else:
    print('⚠️  Aucun GPU détecté.')
    print('→ Exécution > Modifier le type d\'exécution > GPU (T4)')

PyTorch version : 2.11.0+cu128
CUDA disponible : True
GPU : Tesla T4
VRAM : 15.6 Go


## Cellule 2 — Installation des dépendances

In [2]:
# YOLOv8 via Ultralytics
!pip install ultralytics roboflow -q

# YOLOv5 via GitHub
!git clone https://github.com/ultralytics/yolov5.git
!pip install -r yolov5/requirements.txt -q

# Outils d'analyse
!pip install seaborn pandas matplotlib -q

print('✅ Installation terminée.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.3/260.3 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 136.8 MB/s eta 0:00:00
Cloning into 'yolov5'...
remote: Enumerating objects: 18420, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 18420 (delta 59), reused 29 (delta 29), pack-reused 18333 (from 3)
Receiving objects: 100% (18420/18420), 17.53 MiB | 26.35 MiB/s, done.
Resolving deltas: 100% (12512/12512), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 5.5 MB/s eta 0:00:00
✅ Installation terminée.


## Cellule 3 — Montage Google Drive

In [3]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Dossier de sauvegarde des résultats sur Drive
SAVE_DIR = '/content/drive/MyDrive/yolo_project'
RESULTS_DIR = f'{SAVE_DIR}/results'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(f'{SAVE_DIR}/figures', exist_ok=True)

print(f'✅ Drive monté. Résultats dans : {RESULTS_DIR}')

Mounted at /content/drive
✅ Drive monté. Résultats dans : /content/drive/MyDrive/yolo_project/results


## Cellule 4 — Téléchargement du dataset Pascal VOC

In [4]:
from google.colab import drive
import zipfile, glob, os

drive.mount('/content/drive')


ZIP_PATH = '/content/drive/MyDrive/Pascal VOC 2012.v1-raw.yolov5pytorch.zip'

print("Extraction...")
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content/pascal_voc')

yaml_files = glob.glob('/content/pascal_voc/**/data.yaml', recursive=True)
DATA_YAML = yaml_files[0]
print(f"✅ data.yaml : {DATA_YAML}")
with open(DATA_YAML) as f:
    print(f.read())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extraction...
✅ data.yaml : /content/pascal_voc/data.yaml
train: ../train/images
val: ../valid/images

nc: 20
names: ['aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor']


In [5]:
import yaml
import glob

# Trouver les dossiers d'images réels dans le dataset téléchargé
train_path = glob.glob(f'/content/pascal_voc/**/train/images', recursive=True)
valid_path = glob.glob(f'/content/pascal_voc/**/valid/images', recursive=True)
test_path  = glob.glob(f'/content/pascal_voc/**/test/images',  recursive=True)

print(f'train : {train_path}')
print(f'valid : {valid_path}')
print(f'test  : {test_path}')

with open(DATA_YAML, 'r') as f:
    content = yaml.safe_load(f)

# Remplacer par des chemins absolus (et supprimer 'path' pour éviter tout conflit)
content.pop('path', None)
content['train'] = train_path[0] if train_path else content.get('train')
content['val']   = valid_path[0] if valid_path else content.get('val')
if test_path:
    content['test'] = test_path[0]

with open(DATA_YAML, 'w') as f:
    yaml.dump(content, f)

print(f'\n✅ data.yaml corrigé : {DATA_YAML}')
with open(DATA_YAML) as f:
    print(f.read())

train : ['/content/pascal_voc/train/images']
valid : ['/content/pascal_voc/valid/images']
test  : []

✅ data.yaml corrigé : /content/pascal_voc/data.yaml
names:
- aeroplane
- bicycle
- bird
- boat
- bottle
- bus
- car
- cat
- chair
- cow
- diningtable
- dog
- horse
- motorbike
- person
- pottedplant
- sheep
- sofa
- train
- tvmonitor
nc: 20
train: /content/pascal_voc/train/images
val: /content/pascal_voc/valid/images



## Cellule 5 — Configuration partagée

In [6]:
import time
import json
import csv
from pathlib import Path

# Paramètres identiques pour les deux modèles
CONFIG = {
    'img_size'  : 640,
    'epochs'    : 50,     # plafond max ; early stopping arrêtera avant si possible
    'batch_size': 32,     # 16 -> 32 : T4 (16 Go) supporte largement 32 en 640px, entraînement ~2x plus rapide
    'device'    : 0,      # GPU
    'patience'  : 10,     # early stopping : arrêt si pas d'amélioration après N époques
    'workers'   : 8,      # threads de chargement des données (accélère le dataloader)
}

AUGMENTATION_STRATEGIES = [
    'no_augmentation',
    'flip_horizontal',
    'rotation',
    'mosaic',
    'mixup',
]

# Fichier CSV global pour tous les résultats
ALL_RESULTS_CSV = f'{SAVE_DIR}/all_results.csv'

def save_run_metrics(model, augmentation, metrics, elapsed_min):
    """Sauvegarde les métriques d'un run dans le CSV global."""
    row = {
        'model'          : model,
        'augmentation'   : augmentation,
        'map50'          : round(metrics.get('map50', 0), 4),
        'map50_95'       : round(metrics.get('map50_95', 0), 4),
        'precision'      : round(metrics.get('precision', 0), 4),
        'recall'         : round(metrics.get('recall', 0), 4),
        'f1'             : round(metrics.get('f1', 0), 4),
        'train_time_min' : round(elapsed_min, 1),
    }
    file_exists = Path(ALL_RESULTS_CSV).exists()
    with open(ALL_RESULTS_CSV, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)
    print(f'  → Métriques sauvegardées dans {ALL_RESULTS_CSV}')
    return row

print('✅ Configuration prête.')
print(f'   Modèle     : YOLOv5s + YOLOv8s')
print(f'   Époques    : {CONFIG["epochs"]} (patience={CONFIG["patience"]})')
print(f'   Résolution : {CONFIG["img_size"]}×{CONFIG["img_size"]}')
print(f'   Batch      : {CONFIG["batch_size"]}')


✅ Configuration prête.
   Modèle     : YOLOv5s + YOLOv8s
   Époques    : 50 (patience=10)
   Résolution : 640×640
   Batch      : 32


In [7]:
import os
for aug in ['no_augmentation', 'flip_horizontal']:
    path = f'{RESULTS_DIR}/yolov5s_{aug}/weights/last.pt'
    print(aug, '→', 'existe' if os.path.exists(path) else 'MANQUANT')

no_augmentation → existe
flip_horizontal → existe


## Cellule 6 — Entraînements YOLOv5 (5 runs)
> Durée estimée : ~45–70 min par run sur T4 → ~4–5h au total

In [ ]:
import yaml
import pandas as pd
import os
import time

os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'disabled'

def get_yolov5_hyp(augmentation):
    """Génère les hyperparamètres YOLOv5 pour une stratégie d'augmentation."""
    base = {
        'lr0': 0.01, 'lrf': 0.01, 'momentum': 0.937, 'weight_decay': 0.0005,
        'warmup_epochs': 3.0, 'warmup_momentum': 0.8, 'warmup_bias_lr': 0.1,
        'box': 0.05, 'cls': 0.5, 'cls_pw': 1.0, 'obj': 1.0, 'obj_pw': 1.0,
        'iou_t': 0.20, 'anchor_t': 4.0, 'fl_gamma': 0.0,
        'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0,
        'degrees': 0.0, 'translate': 0.0, 'scale': 0.0, 'shear': 0.0,
        'perspective': 0.0, 'flipud': 0.0, 'fliplr': 0.0,
        'mosaic': 0.0, 'mixup': 0.0, 'copy_paste': 0.0,
    }
    if augmentation == 'flip_horizontal': base['fliplr']  = 0.5
    elif augmentation == 'rotation'     : base['degrees'] = 10.0
    elif augmentation == 'mosaic'       : base['mosaic']  = 1.0
    elif augmentation == 'mixup'        : base['mixup']   = 0.1
    return base

def parse_yolov5_csv(run_dir):
    """Lit les métriques finales depuis results.csv de YOLOv5."""
    csv_path = f'{run_dir}/results.csv'
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    last = df.iloc[-1]
    def g(cols):
        for c in cols:
            if c in df.columns: return float(last[c])
        return 0.0
    p = g(['metrics/precision'])
    r = g(['metrics/recall'])
    return {
        'map50'    : g(['metrics/mAP_0.5']),
        'map50_95' : g(['metrics/mAP_0.5:0.95']),
        'precision': p,
        'recall'   : r,
        'f1'       : 2*p*r/(p+r) if (p+r) > 0 else 0.0,
    }

def is_run_complete(run_dir, target_epochs_default):
    """Vérifie si un run est allé jusqu'au bout, en lisant SA PROPRE config (opt.yaml)."""
    csv_path = f'{run_dir}/results.csv'
    opt_path = f'{run_dir}/opt.yaml'
    if not os.path.exists(csv_path):
        return False
    target_epochs = target_epochs_default
    if os.path.exists(opt_path):
        with open(opt_path) as f:
            opt = yaml.safe_load(f)
        target_epochs = opt.get('epochs', target_epochs)
    try:
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
        return len(df) >= target_epochs
    except Exception:
        return False

# ─── BOUCLE D'ENTRAÎNEMENT YOLOV5 (avec reprise) ───
yolov5_results = {}
for aug in AUGMENTATION_STRATEGIES:
    print(f'\n{"="*55}')
    print(f'YOLOv5s | Augmentation : {aug}')
    print(f'{"="*55}')

    run_name = f'yolov5s_{aug}'
    run_dir  = f'{RESULTS_DIR}/{run_name}'
    last_pt  = f'{run_dir}/weights/last.pt'

    # Cas 1 : run déjà terminé -> on saute, on récupère les métriques existantes
    if is_run_complete(run_dir, CONFIG['epochs']):
        print(f'  ✅ Déjà terminé, on saute ce run.')
        try:
            metrics = parse_yolov5_csv(run_dir)
            row = save_run_metrics('yolov5s', aug, metrics, 0.0)
            yolov5_results[aug] = row
            print(f'  mAP@0.5      : {metrics["map50"]:.4f}')
            print(f'  mAP@0.5:0.95 : {metrics["map50_95"]:.4f}')
        except Exception as e:
            print(f'  ⚠️  Erreur lecture métriques existantes : {e}')
        continue

    t0 = time.time()

    # Cas 2 : run interrompu en cours -> reprise via --resume
    if os.path.exists(last_pt):
        print(f'  ↻ Checkpoint trouvé, reprise depuis {last_pt}')
        !python yolov5/train.py --resume {last_pt}
    # Cas 3 : run jamais commencé -> lancement normal
    else:
        print(f'  ▶ Nouveau run.')
        hyp = get_yolov5_hyp(aug)
        hyp_path = f'/content/hyp_{aug}.yaml'
        with open(hyp_path, 'w') as f:
            yaml.dump(hyp, f)
        !python yolov5/train.py \
            --img {CONFIG['img_size']} \
            --batch {CONFIG['batch_size']} \
            --epochs {CONFIG['epochs']} \
            --data {DATA_YAML} \
            --weights yolov5s.pt \
            --hyp {hyp_path} \
            --name {run_name} \
            --project {RESULTS_DIR} \
            --device {CONFIG['device']} \
            --workers {CONFIG['workers']} \
            --patience {CONFIG['patience']} \
            --exist-ok \
            --cache

    elapsed = (time.time() - t0) / 60
    try:
        metrics = parse_yolov5_csv(run_dir)
        row = save_run_metrics('yolov5s', aug, metrics, elapsed)
        yolov5_results[aug] = row
        print(f'  mAP@0.5      : {metrics["map50"]:.4f}')
        print(f'  mAP@0.5:0.95 : {metrics["map50_95"]:.4f}')
        print(f'  Durée        : {elapsed:.1f} min')
    except Exception as e:
        print(f'  ⚠️  Erreur lecture métriques : {e}')

print(f'\n✅ YOLOv5 — Tous les runs terminés ({len(yolov5_results)}/5)')

# Nettoyage des doublons éventuels (si un run a été enregistré plusieurs fois)
df_final = pd.read_csv(ALL_RESULTS_CSV)
df_final = df_final.drop_duplicates(subset=['model', 'augmentation'], keep='last')
df_final.to_csv(ALL_RESULTS_CSV, index=False)
print('\n📊 Résultats finaux (dédoublonnés) :')
print(df_final)


YOLOv5s | Augmentation : no_augmentation
  ✅ Déjà terminé, on saute ce run.
  → Métriques sauvegardées dans /content/drive/MyDrive/yolo_project/all_results.csv
  mAP@0.5      : 0.5083
  mAP@0.5:0.95 : 0.3189

YOLOv5s | Augmentation : flip_horizontal
  ✅ Déjà terminé, on saute ce run.
  → Métriques sauvegardées dans /content/drive/MyDrive/yolo_project/all_results.csv
  mAP@0.5      : 0.5496
  mAP@0.5:0.95 : 0.3484

YOLOv5s | Augmentation : rotation
  ↻ Checkpoint trouvé, reprise depuis /content/drive/MyDrive/yolo_project/results/yolov5s_rotation/weights/last.pt
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://gith

## Cellule 7 — Entraînements YOLOv8 (5 runs)
> Durée estimée : ~50–80 min par run sur T4

In [ ]:
from ultralytics import YOLO

def get_yolov8_aug_params(augmentation):
    """Retourne les paramètres d'augmentation pour YOLOv8."""
    base = {
        'fliplr': 0.0, 'flipud': 0.0, 'degrees': 0.0,
        'translate': 0.0, 'scale': 0.0, 'shear': 0.0,
        'perspective': 0.0, 'mosaic': 0.0, 'mixup': 0.0,
        'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0, 'copy_paste': 0.0,
    }
    if augmentation == 'flip_horizontal': base['fliplr']  = 0.5
    elif augmentation == 'rotation'     : base['degrees'] = 10.0
    elif augmentation == 'mosaic'       : base['mosaic']  = 1.0
    elif augmentation == 'mixup'        : base['mixup']   = 0.1
    return base

# ─── BOUCLE D'ENTRAÎNEMENT YOLOV8 ───
yolov8_results = {}

for aug in AUGMENTATION_STRATEGIES:
    print(f'\n{"="*55}')
    print(f'YOLOv8s | Augmentation : {aug}')
    print(f'{"="*55}')

    aug_params = get_yolov8_aug_params(aug)
    run_name   = f'yolov8s_{aug}'

    model = YOLO('yolov8s.pt')
    t0 = time.time()

    try:
        model.train(
            data    = DATA_YAML,
            epochs  = CONFIG['epochs'],
            imgsz   = CONFIG['img_size'],
            batch   = CONFIG['batch_size'],
            device  = CONFIG['device'],
            workers = CONFIG['workers'],
            patience= CONFIG['patience'],
            name    = run_name,
            project = RESULTS_DIR,
            exist_ok= True,
            cache   = True,
            verbose = True,
            plots   = True,
            **aug_params,
        )

        elapsed  = (time.time() - t0) / 60
        run_path = f'{RESULTS_DIR}/{run_name}'

        # Évaluation sur validation set
        val_results = model.val(data=DATA_YAML, imgsz=640, batch=16, device=0)
        p = float(val_results.box.mp)
        r = float(val_results.box.mr)

        metrics = {
            'map50'    : float(val_results.box.map50),
            'map50_95' : float(val_results.box.map),
            'precision': p,
            'recall'   : r,
            'f1'       : 2*p*r/(p+r) if (p+r) > 0 else 0.0,
        }

        row = save_run_metrics('yolov8s', aug, metrics, elapsed)
        yolov8_results[aug] = row

        print(f'  mAP@0.5      : {metrics["map50"]:.4f}')
        print(f'  mAP@0.5:0.95 : {metrics["map50_95"]:.4f}')
        print(f'  Durée        : {elapsed:.1f} min')

    except Exception as e:
        print(f'  ❌ Erreur run {aug} : {e}')
        import traceback
        traceback.print_exc()

    # Libérer la mémoire GPU entre les runs
    import gc
    del model
    gc.collect()
    torch.cuda.empty_cache()

print(f'\n✅ YOLOv8 — Tous les runs terminés ({len(yolov8_results)}/5)')

## Cellule 8 — Résultats consolidés

In [ ]:
import pandas as pd

df = pd.read_csv(ALL_RESULTS_CSV)
print('=== TABLEAU RÉCAPITULATIF ===')
print(df.to_string(index=False))

print('\n=== MEILLEUR RUN PAR MODÈLE (mAP@0.5:0.95) ===')
for model in df['model'].unique():
    best = df[df['model'] == model].sort_values('map50_95', ascending=False).iloc[0]
    print(f'{model} → {best["augmentation"]} : {best["map50_95"]:.4f}')

print('\n=== GAIN RELATIF YOLOv8 vs YOLOv5 (mAP@0.5:0.95) ===')
for aug in df['augmentation'].unique():
    v5 = df[(df['model']=='yolov5s') & (df['augmentation']==aug)]['map50_95'].values
    v8 = df[(df['model']=='yolov8s') & (df['augmentation']==aug)]['map50_95'].values
    if len(v5) and len(v8) and v5[0] > 0:
        gain = (v8[0] - v5[0]) / v5[0] * 100
        print(f'  {aug:22s}: {gain:+.2f}%')

## Cellule 9 — Graphiques comparatifs

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

AUGS = ['no_augmentation','flip_horizontal','rotation','mosaic','mixup']
AUG_LABELS = ['Baseline','Flip H.','Rotation','Mosaic','MixUp']
COLORS = {'yolov5s': '#1f77b4', 'yolov8s': '#ff7f0e'}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Impact des augmentations — YOLOv5s vs YOLOv8s | Pascal VOC',
             fontsize=14, fontweight='bold')

for ax, metric, label in zip(axes, ['map50', 'map50_95'], ['mAP@0.5', 'mAP@0.5:0.95']):
    x = np.arange(len(AUGS))
    for model in ['yolov5s', 'yolov8s']:
        y = []
        for aug in AUGS:
            val = df[(df['model']==model) & (df['augmentation']==aug)][metric].values
            y.append(val[0] if len(val) else np.nan)
        ax.plot(x, y, marker='o', linewidth=2.5, markersize=8,
                label=model, color=COLORS[model])
        for xi, yi in zip(x, y):
            if not np.isnan(yi):
                ax.annotate(f'{yi:.3f}', (xi, yi),
                            textcoords='offset points', xytext=(0, 10),
                            ha='center', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(AUG_LABELS, rotation=15)
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(f'{SAVE_DIR}/figures/augmentation_impact.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure sauvegardée sur Drive.')

## Cellule 10 — Téléchargement des résultats

In [ ]:
from google.colab import files
import shutil

# Créer une archive ZIP de tous les résultats
shutil.make_archive('/content/yolo_results', 'zip', SAVE_DIR)

# Télécharger l'archive
files.download('/content/yolo_results.zip')

print('✅ Archive téléchargée.')
print('Décompressez-la dans votre projet local et committez sur GitHub :')
print('  cp -r yolo_results/results/* ./results/')
print('  cp -r yolo_results/figures/* ./figures/')
print('  git add results/ figures/')
print('  git commit -m "feat: ajout résultats entraînements GPU"')
print('  git push')